# Induction Motor Fault Detection Using RMS Currents

This notebook is self-contained for Colab. It converts the raw instantaneous current readings into one RMS row per electrical cycle, ignores `I4`, trains the models on the RMS dataset, and saves the trained artifacts.

In [ ]:
!pip install xgboost==2.0.3 pandas scikit-learn joblib tqdm --quiet
print("✓ Dependencies installed")

In [ ]:
import os
import re
import time
import warnings
from dataclasses import dataclass

import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
DEFAULT_LINE_FREQUENCY_HZ = 50.0
np.random.seed(RANDOM_SEED)

CSV_FILES = [
    "Fullload_1_mu_rf3.csv",
    "Fullload_3_mu_Rf3.csv",
    "Fullload_5_mu_Rf3.csv",
    "Fullload_healthy.csv",
    "Halfload_1_mu_Rf3.csv",
    "Halfload_3_mu_Rf3.csv",
    "Halfload_5__rf3.csv",
    "Halfload_healthy.csv",
    "Noload_1_mu_Rf3.csv",
    "Noload_3_mu_Rf3.csv",
    "Noload_5_mu_Rf3.csv",
    "Noload_healthy.csv",
]


@dataclass
class CycleInfo:
    sample_interval_s: float
    samples_per_cycle: int
    frequency_hz: float
    estimation_source: str


def check_gpu():
    try:
        test_model = xgb.XGBClassifier(tree_method="hist", device="cuda", n_estimators=1, verbosity=0)
        test_model.fit(np.random.rand(10, 3), np.random.randint(0, 2, 10))
        print("✓ GPU detected and working!")
        return True
    except Exception as exc:
        print(f"⚠ GPU not available, using CPU: {exc}")
        return False


def extract_labels(filename):
    fn = filename.lower()
    is_faulty = 0 if "healthy" in fn else 1
    severity = 0
    if is_faulty:
        if "_1_" in fn or "1_mu" in fn:
            severity = 1
        elif "_3_" in fn or "3_mu" in fn:
            severity = 2
        elif "_5_" in fn or "_5__" in fn:
            severity = 3

    if "fullload" in fn:
        load_class = 2
    elif "halfload" in fn:
        load_class = 1
    elif "noload" in fn:
        load_class = 0
    else:
        load_class = 0

    return is_faulty, severity, load_class


def find_current_cols(columns):
    cols = list(columns)
    i1 = next((c for c in cols if "i1" in c.lower()), None)
    i2 = next((c for c in cols if "i2" in c.lower()), None)
    i3 = next((c for c in cols if "i3" in c.lower()), None)
    if i1 and i2 and i3:
        return [i1, i2, i3]
    raise ValueError(f"Cannot find I1, I2, I3 in columns: {cols}")


def parse_sample_interval(columns):
    for column in columns:
        match = re.search(r"ts@([0-9.eE+-]+)", column)
        if match:
            return float(match.group(1))
    raise ValueError("Could not parse sample interval from CSV headers")


def _positive_zero_crossing_distances(signal, min_samples, max_samples):
    centered = signal - np.nanmean(signal)
    if len(centered) < 2:
        return np.array([], dtype=np.int32)

    crossings = np.flatnonzero((centered[:-1] < 0.0) & (centered[1:] >= 0.0))
    if len(crossings) < 2:
        return np.array([], dtype=np.int32)

    distances = np.diff(crossings)
    distances = distances[(distances >= min_samples) & (distances <= max_samples)]
    return distances.astype(np.int32)


def estimate_cycle_info(df, current_cols, default_hz=DEFAULT_LINE_FREQUENCY_HZ):
    dt = parse_sample_interval(current_cols)
    min_samples = int(round(1.0 / (100.0 * dt)))
    max_samples = int(round(1.0 / (20.0 * dt)))

    candidate_distances = []
    for column in current_cols:
        signal = pd.to_numeric(df[column], errors="coerce").to_numpy(dtype=np.float64)
        signal = signal[np.isfinite(signal)]
        if len(signal) < max_samples:
            continue
        distances = _positive_zero_crossing_distances(signal, min_samples, max_samples)
        if len(distances):
            candidate_distances.append(distances)

    if candidate_distances:
        distances = np.concatenate(candidate_distances)
        samples_per_cycle = int(np.round(np.median(distances)))
        frequency_hz = 1.0 / (samples_per_cycle * dt)
        return CycleInfo(dt, samples_per_cycle, frequency_hz, "zero_crossing_median")

    samples_per_cycle = int(round(1.0 / (default_hz * dt)))
    frequency_hz = 1.0 / (samples_per_cycle * dt)
    return CycleInfo(dt, samples_per_cycle, frequency_hz, "default_frequency")


def compute_rms_per_cycle(df, current_cols, cycle_info):
    numeric_df = df[current_cols].apply(pd.to_numeric, errors="coerce")
    values = numeric_df.to_numpy(dtype=np.float32)
    valid_rows = np.isfinite(values).all(axis=1)
    values = values[valid_rows]

    total_cycles = len(values) // cycle_info.samples_per_cycle
    if total_cycles == 0:
        raise ValueError("Not enough valid samples to form a full electrical cycle")

    trimmed = values[: total_cycles * cycle_info.samples_per_cycle]
    cycles = trimmed.reshape(total_cycles, cycle_info.samples_per_cycle, 3)
    rms_values = np.sqrt(np.mean(np.square(cycles), axis=1))

    cycle_df = pd.DataFrame(rms_values, columns=["I1_rms", "I2_rms", "I3_rms"])
    cycle_df["cycle_index"] = np.arange(total_cycles, dtype=np.int32)
    return cycle_df


def build_rms_dataset(csv_files, output_csv="processed_data/rms_cycle_dataset.csv"):
    print("\n" + "=" * 60)
    print("BUILDING RMS DATASET")
    print("=" * 60)

    rows = []
    summaries = []

    for fpath in tqdm(csv_files, desc="Converting to RMS"):
        if not os.path.exists(fpath):
            print(f"  ⚠ Not found: {fpath}")
            continue

        filename = os.path.basename(fpath)
        is_faulty, severity, load_class = extract_labels(filename)
        df = pd.read_csv(fpath, on_bad_lines="skip")
        current_cols = find_current_cols(df.columns)
        cycle_info = estimate_cycle_info(df, current_cols)
        cycle_df = compute_rms_per_cycle(df, current_cols, cycle_info)

        cycle_df["source_file"] = filename
        cycle_df["binary"] = is_faulty
        cycle_df["severity"] = severity
        cycle_df["load"] = load_class
        cycle_df["sample_interval_s"] = cycle_info.sample_interval_s
        cycle_df["samples_per_cycle"] = cycle_info.samples_per_cycle
        cycle_df["estimated_frequency_hz"] = cycle_info.frequency_hz
        cycle_df["frequency_source"] = cycle_info.estimation_source

        rows.append(cycle_df)
        summaries.append((filename, len(cycle_df), cycle_info.samples_per_cycle, cycle_info.frequency_hz, cycle_info.estimation_source))

    if not rows:
        raise FileNotFoundError("No usable CSV files were found")

    rms_df = pd.concat(rows, ignore_index=True)
    os.makedirs(os.path.dirname(output_csv), exist_ok=True)
    rms_df.to_csv(output_csv, index=False)

    for filename, cycles, samples_per_cycle, frequency_hz, source in summaries:
        print(
            f"  ✓ {filename}: {cycles:,} cycles | samples/cycle={samples_per_cycle} | "
            f"freq={frequency_hz:.3f} Hz | source={source}"
        )

    print(f"\n✓ RMS dataset saved to '{output_csv}'")
    print(f"✓ Total RMS rows: {len(rms_df):,}")
    return rms_df


def rotate_phases(X, k):
    return X if k == 0 else np.roll(X, -k, axis=1)


def add_features(X):
    i1, i2, i3 = X[:, 0], X[:, 1], X[:, 2]
    i_sum = i1 + i2 + i3
    i_abs_max = np.maximum.reduce([np.abs(i1), np.abs(i2), np.abs(i3)])
    i_abs_min = np.minimum.reduce([np.abs(i1), np.abs(i2), np.abs(i3)])
    i_range = i_abs_max - i_abs_min
    i_std = np.std(X, axis=1)
    return np.column_stack([X, i_sum, i_abs_max, i_abs_min, i_range, i_std])


def expand_split_with_phase_labels(base_df):
    X = base_df[["I1_rms", "I2_rms", "I3_rms"]].to_numpy(dtype=np.float32)
    y_binary, y_severity, y_phase, y_load = [], [], [], []
    X_parts = []

    for row_idx, row in enumerate(base_df.itertuples(index=False)):
        sample = X[row_idx]
        if row.binary == 1:
            for k in range(3):
                X_parts.append(rotate_phases(sample.reshape(1, -1), k)[0])
                y_binary.append(1)
                y_severity.append(int(row.severity))
                y_phase.append(k + 1)
                y_load.append(int(row.load))
        else:
            X_parts.append(sample)
            y_binary.append(0)
            y_severity.append(0)
            y_phase.append(0)
            y_load.append(int(row.load))

    X_expanded = np.asarray(X_parts, dtype=np.float32)
    y_dict = {
        "binary": np.asarray(y_binary, dtype=np.int32),
        "severity": np.asarray(y_severity, dtype=np.int32),
        "phase": np.asarray(y_phase, dtype=np.int32),
        "load": np.asarray(y_load, dtype=np.int32),
    }
    return X_expanded, y_dict


def split_rms_dataset(rms_df):
    idx = np.arange(len(rms_df))
    train_idx, test_idx = train_test_split(idx, test_size=0.1, random_state=RANDOM_SEED, stratify=rms_df["binary"])
    train_idx, val_idx = train_test_split(train_idx, test_size=0.222, random_state=RANDOM_SEED, stratify=rms_df.iloc[train_idx]["binary"])
    return (
        rms_df.iloc[train_idx].reset_index(drop=True),
        rms_df.iloc[val_idx].reset_index(drop=True),
        rms_df.iloc[test_idx].reset_index(drop=True),
    )


def train_models_from_rms(rms_df, use_gpu=True):
    print("\n" + "=" * 60)
    print("TRAINING ON RMS FEATURES")
    print("=" * 60)

    train_df, val_df, test_df = split_rms_dataset(rms_df)
    X_train, y_train = expand_split_with_phase_labels(train_df)
    X_val, y_val = expand_split_with_phase_labels(val_df)
    X_test, y_test = expand_split_with_phase_labels(test_df)

    print(f"Base cycles -> Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
    print(f"Expanded rows -> Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")

    X_train = add_features(X_train)
    X_val = add_features(X_val)
    X_test = add_features(X_test)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train).astype(np.float32)
    X_val = scaler.transform(X_val).astype(np.float32)
    X_test = scaler.transform(X_test).astype(np.float32)

    tasks = [
        ("binary", ["Healthy", "Faulty"]),
        ("severity", ["Healthy", "1μ", "3μ", "5μ"]),
        ("phase", ["Healthy", "Phase 1", "Phase 2", "Phase 3"]),
        ("load", ["No Load", "Half Load", "Full Load"]),
    ]

    models = {}
    results = {}

    for task, names in tasks:
        print(f"\n--- {task.upper()} ---")
        print(f"  Classes in training: {np.unique(y_train[task])}")

        base_params = {
            "n_estimators": 200,
            "max_depth": 8,
            "learning_rate": 0.1,
            "min_child_weight": 50,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "reg_lambda": 1.0,
            "random_state": RANDOM_SEED,
            "verbosity": 0,
            "tree_method": "hist",
            "device": "cuda" if use_gpu else "cpu",
        }

        if len(names) == 2:
            base_params["objective"] = "binary:logistic"
        else:
            base_params["objective"] = "multi:softmax"
            base_params["num_class"] = len(names)

        model = xgb.XGBClassifier(**base_params)
        t0 = time.time()
        model.fit(X_train, y_train[task], eval_set=[(X_val, y_val[task])], verbose=False)
        train_time = time.time() - t0

        val_acc = accuracy_score(y_val[task], model.predict(X_val))
        test_acc = accuracy_score(y_test[task], model.predict(X_test))
        print(f"  Time: {train_time:.1f}s | Val: {val_acc * 100:.2f}% | Test: {test_acc * 100:.2f}%")

        models[task] = model
        results[task] = {"val": val_acc, "test": test_acc}

    return models, scaler, results


def save_models(models, scaler, output_dir="trained_models"):
    os.makedirs(output_dir, exist_ok=True)
    joblib.dump(scaler, f"{output_dir}/scaler.joblib")
    for name, model in models.items():
        model.save_model(f"{output_dir}/model_{name}.json")
        joblib.dump(model, f"{output_dir}/model_{name}.joblib")
    print(f"\n✓ Models saved to '{output_dir}/'")


use_gpu = check_gpu()
existing = [f for f in CSV_FILES if os.path.exists(f)]
print(f"\nFound {len(existing)}/{len(CSV_FILES)} files")

if len(existing) == 0:
    raise FileNotFoundError("No files found. Upload the CSV files to Colab first.")

rms_df = build_rms_dataset(existing)
models, scaler, results = train_models_from_rms(rms_df, use_gpu=use_gpu)
save_models(models, scaler)

print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print("\n┌─────────────────┬────────────┬────────────┐")
print("│ Task            │ Validation │ Test       │")
print("├─────────────────┼────────────┼────────────┤")
for task in ["binary", "severity", "phase", "load"]:
    v, t = results[task]["val"] * 100, results[task]["test"] * 100
    print(f"│ {task:<15} │ {v:>8.2f}%  │ {t:>8.2f}%  │")
print("└─────────────────┴────────────┴────────────┘")

rms_df.head()

In [ ]:
from google.colab import files

!zip -r trained_models.zip trained_models processed_data
files.download('trained_models.zip')

In [ ]:
scaler = joblib.load('trained_models/scaler.joblib')
model_binary = joblib.load('trained_models/model_binary.joblib')
model_severity = joblib.load('trained_models/model_severity.joblib')
model_phase = joblib.load('trained_models/model_phase.joblib')
model_load = joblib.load('trained_models/model_load.joblib')

# These must be RMS currents from exactly one electrical cycle.
I1_rms, I2_rms, I3_rms = 2.20, 2.10, 2.05

X = np.array([[I1_rms, I2_rms, I3_rms]], dtype=np.float32)
I_sum = X[:, 0] + X[:, 1] + X[:, 2]
I_abs_max = np.maximum.reduce([np.abs(X[:, 0]), np.abs(X[:, 1]), np.abs(X[:, 2])])
I_abs_min = np.minimum.reduce([np.abs(X[:, 0]), np.abs(X[:, 1]), np.abs(X[:, 2])])
I_range = I_abs_max - I_abs_min
I_std = np.std(X, axis=1)
X_features = np.column_stack([X, I_sum, I_abs_max, I_abs_min, I_range, I_std])
X_scaled = scaler.transform(X_features)

pred_binary = int(model_binary.predict(X_scaled)[0])
pred_severity = int(model_severity.predict(X_scaled)[0])
pred_phase = int(model_phase.predict(X_scaled)[0])
pred_load = int(model_load.predict(X_scaled)[0])

binary_labels = {0: 'Healthy', 1: 'Faulty'}
severity_labels = {0: 'Healthy', 1: '1μ', 2: '3μ', 3: '5μ'}
phase_labels = {0: 'Healthy', 1: 'Phase 1', 2: 'Phase 2', 3: 'Phase 3'}
load_labels = {0: 'No Load', 1: 'Half Load', 2: 'Full Load'}

print(f'Input RMS currents: I1={I1_rms:.4f}, I2={I2_rms:.4f}, I3={I3_rms:.4f}')
print(f'Binary:   {binary_labels[pred_binary]}')
print(f'Severity: {severity_labels[pred_severity]}')
print(f'Phase:    {phase_labels[pred_phase]}')
print(f'Load:     {load_labels[pred_load]}')